<a href="https://colab.research.google.com/github/Cartoon1226/Safety_rule_cheking-NLP-/blob/main/Safety_rule_checking_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers nltk pandas numpy scikit-learn


In [2]:
import numpy as np
import pandas as pd
import nltk
import re
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    return " ".join(tokens)


In [5]:
safety_rules = [
    "Workers must wear safety helmets at all times inside the construction site",
    "Gas concentration must be tested before entering confined spaces",
    "Employees must not operate heavy machinery without authorization",
    "Protective gloves are mandatory while handling hazardous chemicals",
    "Emergency exits should not be blocked under any circumstances",
    "Workers must wear safety harnesses when working at heights",
    "Electrical equipment must be switched off before maintenance",
]

# Preprocess rules
processed_rules = [preprocess_text(rule) for rule in safety_rules]


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
rule_embeddings = model.encode(
    processed_rules,
    convert_to_tensor=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
input_logs = [
    "Worker entered confined area without gas testing",
    "Maintenance staff repaired electrical panel while power was on",
    "Employee wore helmet and gloves during chemical handling",
    "Emergency exit was blocked by equipment",
    "Authorized operator used crane with safety harness"
]

processed_logs = [preprocess_text(log) for log in input_logs]


In [9]:
log_embeddings = model.encode(
    processed_logs,
    convert_to_tensor=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
similarity_matrix = util.cos_sim(log_embeddings, rule_embeddings)


In [11]:
THRESHOLD = 0.60

results = []

for i, log in enumerate(input_logs):
    log_result = {
        "Safety Log": log,
        "Violation": "No",
        "Matched Rule": "None",
        "Similarity Score": 0.0,
        "Risk Level": "Low"
    }

    max_score = float(similarity_matrix[i].max())
    max_index = int(similarity_matrix[i].argmax())

    if max_score >= THRESHOLD:
        log_result["Violation"] = "Yes"
        log_result["Matched Rule"] = safety_rules[max_index]
        log_result["Similarity Score"] = round(max_score, 3)

        if max_score > 0.8:
            log_result["Risk Level"] = "Critical"
        elif max_score > 0.7:
            log_result["Risk Level"] = "High"
        else:
            log_result["Risk Level"] = "Medium"

    results.append(log_result)


In [12]:
df_results = pd.DataFrame(results)
df_results


,Safety Log,Violation,Matched Rule,Similarity Score,Risk Level
0,Worker entered confined area without gas testing,Yes,Gas concentration must be tested before enteri...,0.652,Medium
1,Maintenance staff repaired electrical panel wh...,No,None,0.000,Low
2,Employee wore helmet and gloves during chemica...,Yes,Protective gloves are mandatory while handling...,0.724,High
3,Emergency exit was blocked by equipment,Yes,Emergency exits should not be blocked under an...,0.790,High
4,Authorized operator used crane with safety har...,No,None,0.000,Low


In [13]:
for _, row in df_results.iterrows():
    print("\n-----------------------------")
    print("Safety Log :", row["Safety Log"])
    print("Violation  :", row["Violation"])
    print("Matched Rule :", row["Matched Rule"])
    print("Similarity Score :", row["Similarity Score"])
    print("Risk Level :", row["Risk Level"])



-----------------------------
Safety Log : Worker entered confined area without gas testing
Violation  : Yes
Matched Rule : Gas concentration must be tested before entering confined spaces
Similarity Score : 0.652
Risk Level : Medium

-----------------------------
Safety Log : Maintenance staff repaired electrical panel while power was on
Violation  : No
Matched Rule : None
Similarity Score : 0.0
Risk Level : Low

-----------------------------
Safety Log : Employee wore helmet and gloves during chemical handling
Violation  : Yes
Matched Rule : Protective gloves are mandatory while handling hazardous chemicals
Similarity Score : 0.724
Risk Level : High

-----------------------------
Safety Log : Emergency exit was blocked by equipment
Violation  : Yes
Matched Rule : Emergency exits should not be blocked under any circumstances
Similarity Score : 0.79
Risk Level : High

-----------------------------
Safety Log : Authorized operator used crane with safety harness
Violation  : No
Matched 

In [14]:
print("\nSystem Characteristics:")
print("• Zero-shot learning")
print("• No labeled dataset used")
print("• Semantic understanding via embeddings")
print("• Threshold-based decision making")



System Characteristics:
• Zero-shot learning
• No labeled dataset used
• Semantic understanding via embeddings
• Threshold-based decision making


In [15]:
df_results.to_csv("safety_violation_results.csv", index=False)
print("Results saved successfully.")


Results saved successfully.
